# Dự báo nhiệt độ Seq2Seq Jena — Huấn luyện trên Google Colab

> Notebook cho dự án `jena-seq2seq` (đề bài: Sequence-to-Sequence Weather Forecasting trên Jena Climate).

## Nội dung thực hiện

1. **Cài đặt** môi trường + clone repo + tải dữ liệu Jena.
2. **Quan sát dữ liệu**: kiểm tra chia tập, đường cơ sở `persistence` và `seasonal-naive`.
3. **Sweep 15 epoch**: thử nhiều cấu hình siêu tham số để tìm cấu hình tốt nhất.
4. **Train 30 epoch** bằng cấu hình tốt nhất tìm được.
5. **Quan sát quá trình train**: biểu đồ loss/MAE theo epoch, học tốc độ học, teacher forcing, độ trễ suy luận.
6. **Đánh giá** trên tập kiểm thử 2016: MAE/RMSE tại tầm 1/24/72/144, so sánh các kiến trúc.

## Yêu cầu

- Chạy trên **GPU** (Colab T4 hoặc cao hơn) để huấn luyện 3 kiến trúc xong trong thời gian hợp lý. Nếu không có GPU, notebook vẫn chạy nhưng lâu hơn nhiều.
- Dữ liệu Jena được tải tự động từ Kaggle hoặc link trực tiếp bên dưới.


In [ ]:
# === 1. Cài đặt môi trường ===

# KHÔNG cài lại torch: Colab đã có sẵn torch build sẵn cho CUDA. Cài lại sẽ
# làm mất hỗ trợ GPU và notebook sẽ chạy rất chậm.
!pip -q install -q pandas numpy scikit-learn matplotlib PyYAML joblib seaborn

import torch
print("torch:", torch.__version__)
print("cuda khả dụng:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Dung lượng GPU:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")
else:
    print("CẢNH BÁO: không có GPU. Notebook sẽ chạy được nhưng rất lâu.")


In [ ]:
# === 2. Clone repo `Sper-il/COU` ===

import os
import subprocess

REPO_URL = "https://github.com/Sper-il/COU.git"
WORKDIR = "/content/COU"

if not os.path.isdir(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORKDIR], check=True)
else:
    subprocess.run(["git", "-C", WORKDIR, "pull"], check=False)

os.chdir(WORKDIR)
print("thư mục hiện tại:", os.getcwd())
print(sorted(os.listdir("."))[:20])


In [ ]:
# === 3. Tải dữ liệu Jena ===

import subprocess
import urllib.request
from pathlib import Path

DATA_DIR = Path("data/raw/jena_climate_2009_2016.csv")
CSV_PATH = DATA_DIR / "jena_climate_2009_2016.csv"

if CSV_PATH.exists():
    print("Đã có CSV trong repo:", CSV_PATH, CSV_PATH.stat().st_size // (1024 * 1024), "MB")
else:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    # Bộ Jena Climate gốc của đề bài (kaggle: punnenga/jena-climate)
    url = "https://archive.org/download/jena-climate/jena_climate_2009_2016.csv"
    print("Đang tải từ", url, "…")
    urllib.request.urlretrieve(url, CSV_PATH)
    print("Tải xong")

assert CSV_PATH.exists(), f"Không tìm thấy {CSV_PATH}"
print("Dung lượng:", CSV_PATH.stat().st_size // (1024 * 1024), "MB")


In [ ]:
# === 4. Cấu hình chung cho cả chế độ sweep và chế độ train chính ===

import json
import os
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml

# Thư mục lưu kết quả (bản clone mới chưa có)
Path("results").mkdir(exist_ok=True)

# `display` có sẵn trong Colab/Jupyter; thêm bản dự phòng cho môi trường khác.
try:
    from IPython.display import display  # noqa: F401
except Exception:  # noqa: BLE001
    def display(*args, **kwargs):
        """In ra console khi không có IPython."""
        for obj in args:
            print(obj)

from src.data.dataset import build_dataloaders, get_scaler
from src.data.split import load_config
from src.models.architecture import build_model, count_parameters
from src.models.loss import mae_loss, mse_loss
from src.training.scheduler import build_scheduler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.unicode_minus"] = False

# Ngưỡng nghiệm thu lấy từ docs/description.md mục 6
THRESHOLDS = {"mae": 1.5, "rmse": 2.0, "mae_h24": 1.2, "mae_h144": 2.2}
HORIZONS = (1, 24, 72, 144)

# Mốc đường cơ sở đo từ EDA (results/eda/eda_summary.json) — dùng để tham chiếu
PERSISTENCE_2016 = 2.8058
SEASONAL_2016 = 2.4747


def set_seed(seed: int = SEED) -> None:
    """Cố định hạt giống cho python / numpy / torch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_data(data_config="configs/data_config.yaml"):
    """Dựng 3 DataLoader train/val/test theo cấu hình trong repo."""
    cfg = load_config(data_config)
    cfg["batch_size"] = 64
    cfg["num_workers"] = 2
    cfg["seed"] = SEED
    return build_dataloaders(cfg)


print("Thiết bị:", DEVICE)
print("Torch:", torch.__version__)


In [ ]:
# === 5. Dựng DataLoader và kiểm tra chia tập ===

t0 = time.time()
LOADERS = load_data()
SCALER = get_scaler()
print(f"Dựng DataLoader xong sau {time.time() - t0:.1f}s")

for name in ("train", "val", "test"):
    ds = LOADERS[name].dataset
    print(f"{name:>5}: {len(ds):>6} mẫu | {len(LOADERS[name]):>4} lô")

sample = next(iter(LOADERS["train"]))
print("shape một lô:", sample["input"].shape, "->", sample["target"].shape)
print("F = 15 (14 biến gốc + wd_sin + wd_cos)")


In [ ]:
# === 6. Đường cơ sở: persistence và seasonal-naive (mốc sàn để mô hình vượt) ===

from src.evaluation.metrics import compute_persistence_baseline, compute_seasonal_naive, inverse_transform_target


def collect(loader):
    """ Gom (input, target, pred) của một DataLoader về numpy. """
    xs, ys = [], []
    for batch in loader:
        x, y = batch["input"], batch["target"]
        xs.append(x.numpy())
        ys.append(y.numpy())
    return np.concatenate(xs, 0), np.concatenate(ys, 0)


def baseline_report(loader, name):
    """Tính MAE của persistence và seasonal-naive trên một tập (đơn vị °C gốc)."""
    inputs, targets = collect(loader)
    _, targets_c = inverse_transform_target(targets, targets, SCALER)
    pers = compute_persistence_baseline(inputs, targets_c, SCALER)
    seas = compute_seasonal_naive(inputs, targets_c, SCALER)
    print(f"[{name}] persistence = {pers:.4f} °C | seasonal-naive = {seas:.4f} °C")
    return {"persistence_mae": pers, "seasonal_naive_mae": seas}


BASE_TEST = baseline_report(LOADERS["test"], "test 2016")
BASE_VAL = baseline_report(LOADERS["val"], "val 2015")
print("Mục tiêu: mô hình phải vượt cả hai đường cơ sở trên tập kiểm thử 2016.")


In [ ]:
# === 7. Hàm huấn luyện có đầy đủ công cụ quan sát ===

import torch.nn as nn


def train_loop(model, loaders, cfg, run_name="run", verbose=True, log_every_batch=False):
    """Huấn luyện 1 kiến trúc theo cấu hình `cfg` và trả về lịch sử chi tiết.

    Mọi thứ cần để quan sát quá trình train đều được ghi vào `history`:
    loss/MAE huấn luyện và kiểm định theo từng epoch, tỉ lệ teacher forcing,
    tốc độ học, thời gian mỗi epoch, và độ trễ suy luận.

    Args:
        model: mô hình PyTorch đã dựng.
        loaders: dict {train, val, test} DataLoader.
        cfg: dict cấu hình (xem `DEFAULT_CFG`).
        run_name: tên hiển thị.
        verbose: in tiến độ mỗi epoch.
        log_every_batch: in loss từng lô (để thấy độ trễ trong epoch).

    Returns:
        dict gồm `history` (list theo epoch) và `best` (epoch tốt nhất).
    """
    set_seed(cfg.get("seed", SEED))
    model = model.to(DEVICE)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=cfg["lr"], weight_decay=cfg.get("weight_decay", 1e-5)
    )
    scheduler = build_scheduler(optimizer, cfg.get("scheduler", {}))
    criterion = mse_loss

    epochs = cfg["epochs"]
    tf_start = cfg.get("teacher_forcing", 0.5)
    tf_decay_epochs = cfg.get("teacher_forcing_decay_epochs", 10)
    tf_warmup = cfg.get("teacher_forcing_warmup_epochs", 1)
    clip = cfg.get("clip", 1.0)
    patience = cfg.get("early_stop", 5)

    history = []
    best_val = float("inf")
    best_epoch = 0
    best_state = None
    bad_epochs = 0

    for epoch in range(1, epochs + 1):
        # Lịch teacher forcing: giữ ở mức khởi đầu ở epoch đầu, sau đó giảm
        # TUYẾN TÍNH về 0. Khi về 0, mô hình học đúng chế độ tự hồi tiếp như
        # lúc suy luận nên không còn "sốc" (exposure bias).
        if epoch <= tf_warmup:
            tf_ratio = tf_start
        else:
            span = max(tf_decay_epochs, 1)
            progress = min((epoch - tf_warmup) / span, 1.0)
            tf_ratio = max(tf_start * (1.0 - progress), cfg.get("teacher_forcing_min", 0.0))

        # ---- vòng huấn luyện ----
        model.train()
        t0 = time.time()
        tot_loss = 0.0
        tot_mae = 0.0
        n_batch = 0
        for batch in loaders["train"]:
            x, y = batch["input"].to(DEVICE), batch["target"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            pred = model(x, y_in=y, teacher_forcing_ratio=tf_ratio)
            loss = criterion(pred, y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
            tot_loss += float(loss.item())
            tot_mae += float(mae_loss(pred, y).item())
            n_batch += 1
            if log_every_batch and n_batch % 50 == 0:
                print(f"    lô {n_batch:>4} | loss {tot_loss / n_batch:.5f} "
                      f"| {time.time() - t0:.0f}s")
        train_time = time.time() - t0

        # ---- kiểm định ở chế độ tự hồi tiếp (giống hệt lúc suy luận) ----
        model.eval()
        t1 = time.time()
        v_loss = 0.0
        v_mae = 0.0
        n_val = 0
        with torch.no_grad():
            for batch in loaders["val"]:
                x, y = batch["input"].to(DEVICE), batch["target"].to(DEVICE)
                out = model(x, y_in=None)
                v_loss += float(criterion(out, y).item())
                v_mae += float(mae_loss(out, y).item())
                n_val += 1
        val_time = time.time() - t1
        v_loss /= max(n_val, 1)
        v_mae /= max(n_val, 1)

        scheduler.step(v_loss)
        lr_now = optimizer.param_groups[0]["lr"]

        history.append({
            "epoch": epoch,
            "train_loss": tot_loss / max(n_batch, 1),
            "train_mae": tot_mae / max(n_batch, 1),
            "val_loss": v_loss,
            "val_mae": v_mae,
            "teacher_forcing": tf_ratio,
            "lr": lr_now,
            "train_time_s": train_time,
            "val_time_s": val_time,
        })

        improved = v_loss < best_val
        if improved:
            best_val = v_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1

        if verbose:
            mark = "  <-- tot nhat" if improved else ""
            print(f"  Epoch {epoch:>3}/{epochs} | train loss {tot_loss / max(n_batch, 1):.5f} "
                  f"mae {tot_mae / max(n_batch, 1):.5f} | val loss {v_loss:.5f} "
                  f"mae {v_mae:.5f} | tf {tf_ratio:.3f} | lr {lr_now:.2e} "
                  f"| {train_time:.0f}s + {val_time:.0f}s{mark}")

        if bad_epochs >= patience:
            print(f"  DUNG SOM sau {epoch} epoch (val loss khong cai thien {patience} lan)")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return {"history": history, "best_epoch": best_epoch, "best_val_loss": best_val, "model": model}


In [ ]:
# === 8. Đánh giá một mô hình đã huấn luyện trên tập kiểm thử 2016 ===

from src.evaluation.metrics import compute_persistence_baseline, compute_persistence_horizons, compute_seasonal_naive, compute_seasonal_naive_horizons, inverse_transform_target


def evaluate_model(model, loader, scaler=None):
    """Dự báo toàn bộ một tập và trả về chỉ số đầy đủ (đơn vị °C gốc)."""
    model.eval()
    xs, ys, ps = [], [], []
    with torch.no_grad():
        for batch in loader:
            x, y = batch["input"].to(DEVICE), batch["target"].to(DEVICE)
            out = model(x, y_in=None)
            xs.append(x.cpu().numpy())
            ys.append(y.cpu().numpy())
            ps.append(out.cpu().numpy())

    inputs = np.concatenate(xs, 0)
    targets = np.concatenate(ys, 0)
    preds = np.concatenate(ps, 0)
    preds_c, targets_c = inverse_transform_target(preds, targets, scaler)

    m = {
        "mae": float(np.mean(np.abs(preds_c - targets_c))),
        "rmse": float(np.sqrt(np.mean((preds_c - targets_c) ** 2))),
    }
    for h in HORIZONS:
        err = preds_c[:, h - 1] - targets_c[:, h - 1]
        m[f"mae_h{h}"] = float(np.mean(np.abs(err)))
        m[f"rmse_h{h}"] = float(np.sqrt(np.mean(err ** 2)))

    m["persistence_mae"] = compute_persistence_baseline(inputs, targets_c, scaler)
    m.update(compute_persistence_horizons(inputs, targets_c, scaler))
    m["seasonal_naive_mae"] = compute_seasonal_naive(inputs, targets_c, scaler)
    m.update(compute_seasonal_naive_horizons(inputs, targets_c, scaler))
    m["out_of_bounds_count"] = int(np.sum((preds_c < -40) | (preds_c > 50)))
    m["n_samples"] = int(preds.shape[0])
    return m, preds_c, targets_c


def print_scoreboard(m, title=""):
    """In bảng điểm kèm đánh dấu đạt/không đạt theo ngưỡng nghiệm thu."""
    print("=" * 66)
    print(title)
    print("=" * 66)
    for key, label, thr in (
        ("mae", "MAE toan tang", THRESHOLDS["mae"]),
        ("rmse", "RMSE toan tang", THRESHOLDS["rmse"]),
        ("mae_h24", "MAE@24", THRESHOLDS["mae_h24"]),
        ("mae_h144", "MAE@144", THRESHOLDS["mae_h144"]),
    ):
        val = m[key]
        print(f"  {label:<16}{val:8.4f} °C  (<= {thr})  -> "
              f"{'DAT' if val <= thr else 'KHONG DAT'}")
    for key, label in (("persistence_mae", "Persistence"),
                       ("seasonal_naive_mae", "Seasonal-naive")):
        beat = m["mae"] < m[key]
        print(f"  {label:<16}{m[key]:8.4f} °C  -> "
              f"{'DAT (da vuot)' if beat else 'KHONG DAT'}")
    print(f"  MAE theo tam: " + " | ".join(
        f"@{h} = {m[f'mae_h{h}']:.3f}" for h in HORIZONS))
    print("=" * 66)


In [ ]:
# === 10. Bộ công cụ vẽ để quan sát quá trình huấn luyện ===


def plot_training_curves(history, title="Quan sat qua trinh huan luyen", save=None):
    """Vẽ 6 biểu đồ theo dõi: loss, MAE, teacher forcing, lr, thời gian, gap."""
    ep = [h["epoch"] for h in history]
    fig, axes = plt.subplots(2, 3, figsize=(17, 8))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    # 1. hàm mất mát train vs val
    ax = axes[0, 0]
    ax.plot(ep, [h["train_loss"] for h in history], "o-", label="train loss", ms=3)
    ax.plot(ep, [h["val_loss"] for h in history], "s-", label="val loss (tu hoi tiep)", ms=3)
    ax.set_xlabel("epoch")
    ax.set_ylabel("MSE (chu dan hoa)")
    ax.set_title("Ham mat tat train vs val")
    ax.legend()
    ax.grid(alpha=0.3)

    # 2. MAE train vs val
    ax = axes[0, 1]
    ax.plot(ep, [h["train_mae"] for h in history], "o-", label="train MAE", ms=3)
    ax.plot(ep, [h["val_mae"] for h in history], "s-", label="val MAE", ms=3)
    ax.set_xlabel("epoch")
    ax.set_ylabel("MAE (chu dan hoa)")
    ax.set_title("MAE train vs val")
    ax.legend()
    ax.grid(alpha=0.3)

    # 3. tỉ lệ teacher forcing
    ax = axes[0, 2]
    ax.plot(ep, [h["teacher_forcing"] for h in history], "o-", color="darkorange", ms=3)
    ax.set_xlabel("epoch")
    ax.set_ylabel("ty le teacher forcing")
    ax.set_title("Lich giam dan teacher forcing ve 0")
    ax.grid(alpha=0.3)

    # 4. tốc độ học
    ax = axes[1, 0]
    ax.semilogy(ep, [h["lr"] for h in history], "o-", color="purple", ms=3)
    ax.set_xlabel("epoch")
    ax.set_ylabel("learning rate")
    ax.set_title("ReduceLROnPlateau")
    ax.grid(alpha=0.3)

    # 5. thời gian mỗi epoch
    ax = axes[1, 1]
    ax.bar(ep, [h["train_time_s"] for h in history], label="train", color="steelblue")
    ax.bar(ep, [h["val_time_s"] for h in history],
           bottom=[h["train_time_s"] for h in history], label="val", color="orange")
    ax.set_xlabel("epoch")
    ax.set_ylabel("giay")
    ax.set_title("Thoi gian moi epoch")
    ax.legend()
    ax.grid(alpha=0.3, axis="y")

    # 6. khoảng cách train - val (dau hieu qua khớp)
    ax = axes[1, 2]
    gap = [h["train_loss"] - h["val_loss"] for h in history]
    ax.plot(ep, gap, "o-", color="crimson", ms=3)
    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xlabel("epoch")
    ax.set_ylabel("train loss - val loss")
    ax.set_title("Khoang cach train - val (qua khop?)")
    ax.grid(alpha=0.3)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    if save:
        plt.savefig(save, dpi=110, bbox_inches="tight")
    plt.show()


In [ ]:
# === 11. So sánh nhiều cấu hình: biểu đồ + bảng xếp hạng ===


def _col(df, *names):
    """Trả về TÊN cột đầu tiên có trong df trong số các tên đưa ra."""
    for n in names:
        if n in df.columns:
            return n
    raise KeyError(f"Không tìm thấy cột nào trong {names} (có: {list(df.columns)})")


def plot_config_comparison(results):
    """Vẽ biểu đồ so sánh các cấu hình đã thử.

    `results` là list dict. Chấp nhận cả hai kiểu tên cột:
    - dùng trong sweep: `val_mae`, `val_rmse`, `val_mae_h24`, `val_mae_h144`
    - dùng trong đánh giá: `mae`, `rmse`, `mae_h24`, `mae_h144`
    """
    df = pd.DataFrame(results).copy()
    c_mae = _col(df, "mae", "val_mae")
    c_rmse = _col(df, "rmse", "val_rmse")
    df = df.sort_values(c_mae).reset_index(drop=True)

    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    fig.suptitle("So sanh cau hinh sieu tham so (sap xep theo MAE tot nhat)", fontsize=14)

    # 1. MAE / RMSE
    ax = axes[0]
    x = np.arange(len(df))
    ax.bar(x - 0.2, df[c_mae], 0.4, label="MAE toan tang", color="steelblue")
    ax.bar(x + 0.2, df[c_rmse], 0.4, label="RMSE toan tang", color="indianred")
    ax.axhline(THRESHOLDS["mae"], ls="--", color="navy", label="nguong MAE 1.5")
    ax.axhline(THRESHOLDS["rmse"], ls=":", color="darkred", label="nguong RMSE 2.0")
    ax.set_xticks(x)
    ax.set_xticklabels(df["name"], rotation=30, ha="right", fontsize=8)
    ax.set_ylabel("°C")
    ax.set_title("MAE / RMSE theo cau hinh")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, axis="y")

    # 2. MAE theo tầm dự báo
    ax = axes[1]
    for _, row in df.iterrows():
        vals = []
        for h in HORIZONS:
            v = row.get(f"mae_h{h}", row.get(f"val_mae_h{h}", np.nan))
            vals.append(v)
        ax.plot(HORIZONS, vals, "o-", label=row["name"], ms=4)
    ax.set_xlabel("buoc du bao (10 phut)")
    ax.set_ylabel("MAE (°C)")
    ax.set_title("Sai so tich luy theo tam du bao")
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

    # 3. thời gian train (hoặc số tham số)
    ax = axes[2]
    if "train_time_s" in df.columns:
        ax.barh(df["name"], df["train_time_s"], color="teal")
        ax.set_xlabel("giay")
        ax.set_title("Thoi gian huấn luyện (giay)")
    else:
        ax.barh(df["name"], df.get("params", pd.Series(range(len(df)))), color="teal")
        ax.set_xlabel("so tham so")
        ax.set_title("So tham so")
    ax.grid(alpha=0.3, axis="x")

    plt.tight_layout(rect=[0, 0, 1, 0.94])
    plt.show()
    return df


def plot_learning_curves_grid(all_histories):
    """Vẽ đường val loss của mọi cấu hình trên cùng một trục để so sánh tốc độ hội tụ."""
    fig, ax = plt.subplots(figsize=(10, 5))
    for name, res in all_histories.items():
        h = res["history"]
        ax.plot([x["epoch"] for x in h], [x["val_loss"] for x in h], "o-", label=name, ms=3)
    ax.set_xlabel("epoch")
    ax.set_ylabel("val loss (MSE, chuan hoa)")
    ax.set_title("Duong val loss cua cac cau hinh")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
# === 12. Sweep siêu tham số (15 epoch mỗi cấu hình) ===

# Danh sách cấu hình thử. Mỗi cấu hình là một dict ghi đè lên `DEFAULT_CFG`.
# Bắt đầu từ cấu hình gốc trong `configs/train_config.yaml` rồi thử biến thiên
# có kiểm soát để biết yếu tố nào thực sự ảnh hưởng tới chất lượng.
DEFAULT_CFG = {
    "epochs": 15,
    "batch_size": 64,
    "lr": 1e-3,
    "weight_decay": 1e-5,
    "teacher_forcing": 0.5,
    "teacher_forcing_decay_epochs": 5,
    "teacher_forcing_warmup_epochs": 1,
    "teacher_forcing_min": 0.0,
    "clip": 1.0,
    "early_stop": 15,
    "seed": SEED,
    "scheduler": {"lr_factor": 0.5, "lr_patience": 3, "min_lr": 1e-6},
}

# Tham số khởi tạo mô hình theo từng kiến trúc (khớp configs/model_config.yaml)
ARCH_KWARGS = {
    "lstm_s2s": {"input_dim": 15, "hidden_dim": 128, "num_layers": 2, "dropout": 0.1, "pred_len": 144},
    "lstm_attention": {"input_dim": 15, "hidden_dim": 128, "num_layers": 2, "dropout": 0.1, "pred_len": 144},
    "mini_transformer": {"input_dim": 15, "d_model": 64, "nhead": 4, "num_layers": 2, "dropout": 0.1, "pred_len": 144},
}

ARCH_KEYS = ("hidden_dim", "num_layers", "dropout", "d_model", "nhead")


def make_cfg(spec, epochs, early_stop=None, decay_epochs=10):
    """Dựng cấu hình huấn luyện từ một spec của sweep."""
    cfg = dict(DEFAULT_CFG)
    cfg.update({k: v for k, v in spec.items() if k not in ("name", "arch")})
    cfg["epochs"] = epochs
    if early_stop is not None:
        cfg["early_stop"] = early_stop
    cfg["teacher_forcing_decay_epochs"] = decay_epochs
    return cfg


def make_kwargs(arch, cfg):
    """Tham số khởi tạo mô hình theo kiến trúc và các override trong cfg."""
    kwargs = dict(ARCH_KWARGS[arch])
    for key in ARCH_KEYS:
        if key in cfg:
            kwargs[key] = cfg[key]
    return kwargs


# Nhóm 1: quanh cấu hình gốc (baseline LSTM + Attention + Transformer)
# Nhóm 2: biến thiên tốc độ học
# Nhóm 3: biến thiên kiến trúc / kích thước ẩn
SWEEP = [
    {"name": "lstm_s2s_goc", "arch": "lstm_s2s"},
    {"name": "attn_bahdanau_goc", "arch": "lstm_attention"},
    {"name": "transformer_goc", "arch": "mini_transformer"},
    {"name": "lstm_s2s_lr3e-4", "arch": "lstm_s2s", "lr": 3e-4},
    {"name": "lstm_s2s_lr3e-3", "arch": "lstm_s2s", "lr": 3e-3},
    {"name": "attn_lr3e-4", "arch": "lstm_attention", "lr": 3e-4},
    {"name": "attn_hidden64", "arch": "lstm_attention", "hidden_dim": 64},
    {"name": "attn_tf_tu0", "arch": "lstm_attention", "teacher_forcing": 0.0},
    {"name": "attn_tf_decay10", "arch": "lstm_attention", "teacher_forcing_decay_epochs": 10},
]

print(f"Sweep {len(SWEEP)} cau hinh x {DEFAULT_CFG['epochs']} epoch tren {DEVICE}")
print("Lưu ý: chạy sweep trên tập kiểm định để chọn cấu hình, tập kiểm thử 2016 giữ lại để đánh giá một lần cuối.")


In [ ]:
# === 13. Chạy sweep: huấn luyện 15 epoch cho từng cấu hình ===

SWEEP_RESULTS = []
SWEEP_HISTORIES = {}

for spec in SWEEP:
    name = spec["name"]
    arch = spec["arch"]
    cfg = dict(DEFAULT_CFG)
    cfg.update({k: v for k, v in spec.items() if k not in ("name", "arch")})
    kwargs = make_kwargs(arch, cfg)

    print("=" * 70)
    print(f"CAU HINH: {name}  (arch={arch})  |  lr={cfg['lr']}  "
          f"tf={cfg['teacher_forcing']}->{cfg['teacher_forcing_min']}")
    print("=" * 70)

    set_seed(cfg["seed"])
    model = build_model(arch, **kwargs)
    n_params = count_parameters(model)
    print(f"Số tham số: {n_params:,}")

    res = train_loop(model, LOADERS, cfg, run_name=name, verbose=True)
    SWEEP_HISTORIES[name] = res

    # Đánh giá trên tập kiểm định để chọn cấu hình (không dùng tập kiểm thử)
    m_val, _, _ = evaluate_model(res["model"], LOADERS["val"], SCALER)
    total_time = sum(h["train_time_s"] + h["val_time_s"] for h in res["history"])

    row = {
        "name": name,
        "arch": arch,
        "val_mae": m_val["mae"],
        "val_rmse": m_val["rmse"],
        "val_mae_h24": m_val["mae_h24"],
        "val_mae_h144": m_val["mae_h144"],
        "best_epoch": res["best_epoch"],
        "epochs_run": len(res["history"]),
        "params": n_params,
        "train_time_s": round(total_time, 1),
        "lr": cfg["lr"],
        "teacher_forcing": cfg["teacher_forcing"],
    }
    SWEEP_RESULTS.append(row)
    print(f"  >> val MAE = {m_val['mae']:.4f} °C | val RMSE = {m_val['rmse']:.4f} °C "
          f"| tốt nhất ở epoch {res['best_epoch']}")

SWEEP_DF = pd.DataFrame(SWEEP_RESULTS).sort_values("val_mae").reset_index(drop=True)
print("\n" + "=" * 70)
print("BANG XEP HANG THEO VAL MAE (thu nho)")
print("=" * 70)
display(SWEEP_DF[["name", "arch", "val_mae", "val_rmse", "val_mae_h24",
                  "val_mae_h144", "best_epoch", "params", "train_time_s"]])
SWEEP_DF.to_csv("results/sweep_results.csv", index=False, encoding="utf-8")
print("Đã lưu results/sweep_results.csv")


In [ ]:
# === 14. Phân tích kết quả sweep: cấu hình nào tốt nhất? ===

plot_config_comparison(SWEEP_RESULTS)
plot_learning_curves_grid(SWEEP_HISTORIES)

BEST_NAME = SWEEP_DF.iloc[0]["name"]
BEST_ROW = SWEEP_DF.iloc[0]
print(f"Cau hinh tot nhat theo val MAE: {BEST_NAME}")
print(SWEEP_DF.iloc[0].to_string())

# Phân tích thêm: yếu tố nào ảnh hưởng mạnh nhất
print("\n" + "=" * 70)
print("PHAN TICH YEU TO")
print("=" * 70)

for arch in SWEEP_DF["arch"].unique():
    sub = SWEEP_DF[SWEEP_DF["arch"] == arch]
    if len(sub) > 1:
        spread = sub["val_mae"].max() - sub["val_mae"].min()
        print(f"{arch:<18} best {sub['val_mae'].min():.4f} | worst {sub['val_mae'].max():.4f} "
              f"| chenh lech {spread:.4f} °C")

lr_sub = SWEEP_DF[SWEEP_DF["arch"] == "lstm_s2s"]
if len(lr_sub) > 1:
    print(f"\nTac dong cua learning rate (LSTM S2S):")
    for _, r in lr_sub.iterrows():
        print(f"  lr = {r['lr']:.0e} -> val MAE {r['val_mae']:.4f} °C")

# So sanh 3 kien truc o cung cau hinh goc
print("\nSo sanh 3 kien truc (cau hinh goc):")
base = SWEEP_DF[SWEEP_DF["name"].isin(
    ["lstm_s2s_goc", "attn_bahdanau_goc", "transformer_goc"])]
if len(base) == 3:
    print(base[["name", "val_mae", "val_rmse", "params", "train_time_s"]].to_string(index=False))


In [ ]:
# === 15. Huấn luyện chính 30 epoch với cấu hình tốt nhất ===

# Lấy cấu hình thắng trong sweep, nhưng nâng `epochs` lên 30 và giữ nguyên
# mọi siêu tham số khác. `early_stop` bật lại để dừng sớm khi bão hoà.
best_spec = next(s for s in SWEEP if s["name"] == BEST_NAME)
BEST_ARCH = best_spec["arch"]

FINAL_CFG = make_cfg(best_spec, epochs=30, early_stop=5, decay_epochs=10)
FINAL_KWARGS = make_kwargs(BEST_ARCH, FINAL_CFG)

print("=" * 70)
print(f"CHAY CHINH: {BEST_NAME} | arch={BEST_ARCH} | 30 epoch | {DEVICE}")
print("=" * 70)
for k, v in FINAL_CFG.items():
    if k != "scheduler":
        print(f"  {k} = {v}")
print(f"  kwargs mo hinh = {FINAL_KWARGS}")

set_seed(FINAL_CFG["seed"])
final_model = build_model(BEST_ARCH, **FINAL_KWARGS)
print(f"Số tham số: {count_parameters(final_model):,}")

final_res = train_loop(final_model, LOADERS, FINAL_CFG, run_name=BEST_NAME, verbose=True)
final_model = final_res["model"]
print(f"Epoch tốt nhất: {final_res['best_epoch']} | val loss {final_res['best_val_loss']:.5f}")


In [ ]:
# === 16. Quan sát chi tiết quá trình train của lần chạy chính ===

plot_training_curves(final_res["history"], title=f"Quan sat luan chay chinh - {BEST_NAME}")

# Bảng số liệu từng epoch
final_df = pd.DataFrame(final_res["history"])
print("Bang chi tiet tung epoch:")
display(final_df[["epoch", "train_loss", "val_loss", "train_mae", "val_mae",
                  "teacher_forcing", "lr", "train_time_s", "val_time_s"]].round(5))

# Nhận xét tự động dựa trên đường cong
best_h = min(final_res["history"], key=lambda h: h["val_loss"])
last_h = final_res["history"][-1]
train_gap_last = last_h["train_loss"] - last_h["val_loss"]
val_mae_best = best_h["val_mae"]

print("\n" + "=" * 70)
print("DIEN GIAI QUA TRINH TRAIN")
print("=" * 70)
print(f"Epoch tot nhat         : {best_h['epoch']} (val loss {best_h['val_loss']:.5f})")
print(f"Val loss cuoi cung     : {last_h['val_loss']:.5f} sau {len(final_res['history'])} epoch")
print(f"Val MAE tot nhat       : {val_mae_best:.5f} (don vi chuan hoa)")
if train_gap_last < -0.005:
    print("-> Dau hieu QUA KHOP: val loss thap hon train loss, mo hinh dang chua hoa.")
elif train_gap_last > 0.02:
    print("-> Dau hieu QUA KHOP: train loss cao hon val loss, can chinh regularization.")
else:
    print("-> Train va val khop nhau, chua thay hien qua khop ro net.")
if last_h["teacher_forcing"] <= 0.01:
    print("-> Teacher forcing da ve 0: mo hinh da duoc huấn luyện o dung che do tu hoi tiep.")
else:
    print(f"-> Teacher forcing con {last_h['teacher_forcing']:.3f}, chua ve 0 hoan toan.")
if len(final_res["history"]) < FINAL_CFG["epochs"]:
    print(f"-> Da dung som sau {len(final_res['history'])} epoch (val loss khong cai thien).")
else:
    print("-> Da chay het so epoch cho phep.")


In [ ]:
# === 17. Đánh giá cuối trên tập kiểm thử 2016 ===

final_metrics, preds_c, targets_c = evaluate_model(final_model, LOADERS["test"], SCALER)
print_scoreboard(final_metrics, f"KET QUA KIEM THU 2016 - {BEST_NAME}")

# Vẽ mẫu dự báo tốt nhất và xấu nhất theo MAE toàn tầm
per_sample_mae = np.mean(np.abs(preds_c - targets_c), axis=1)
best_i = int(np.argmin(per_sample_mae))
worst_i = int(np.argmax(per_sample_mae))

df_meta = LOADERS["_meta"]["df"]
starts = np.asarray(LOADERS["_meta"]["splits"]["test"])
date_best = df_meta["dt"].iloc[int(starts[best_i]) + 144].strftime("%Y-%m-%d %H:%M")
date_worst = df_meta["dt"].iloc[int(starts[worst_i]) + 144].strftime("%Y-%m-%d %H:%M")

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for ax, idx, title, d in (
    (axes[0], best_i, "Du bao tot nhat (MAE thap nhat)", date_best),
    (axes[1], worst_i, "Du bao xau nhat (MAE cao nhat)", date_worst),
):
    ax.plot(targets_c[idx], label="Thuc te", lw=1.6)
    ax.plot(preds_c[idx], label="Du bao", lw=1.4, ls="--")
    ax.fill_between(range(144), targets_c[idx], preds_c[idx], alpha=0.2, color="orange")
    ax.set_title(f"{title} | {d} | MAE = {per_sample_mae[idx]:.3f} °C")
    ax.set_xlabel("buoc (10 phut)")
    ax.set_ylabel("nhiet do (°C)")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Lưu kết quả
Path("results").mkdir(exist_ok=True)
with open("results/final_metrics.json", "w", encoding="utf-8") as f:
    json.dump(final_metrics, f, ensure_ascii=False, indent=2, default=float)
torch.save({"state_dict": final_model.state_dict(),
            "model_name": BEST_NAME, "arch": BEST_ARCH,
            "model_config": {"arch": BEST_ARCH, "arch_kwargs": FINAL_KWARGS}},
           "results/final_model.pt")
print("Đã lưu results/final_metrics.json và results/final_model.pt")


In [ ]:
# === 18. So sánh 3 kiến trúc trên cùng tập kiểm thử (bảng bắt buộc của đề) ===

# Để so sánh công bằng, huấn luyện mỗi kiến trúc với cùng số epoch và cùng
# cấu hình siêu tham số tốt nhất tìm được ở bước sweep.
COMPARE_ARCHS = ["lstm_s2s", "lstm_attention", "mini_transformer"]
COMPARE_EPOCHS = 15
compare_results = []
compare_models = {}

for arch in COMPARE_ARCHS:
    # Cùng bộ siêu tham số với cấu hình thắng, nhưng tham số riêng của kiến
    # trúc (hidden_dim / d_model) lấy theo mặc định của kiến trúc đó.
    cfg = make_cfg(best_spec, epochs=COMPARE_EPOCHS, early_stop=COMPARE_EPOCHS,
                   decay_epochs=10)
    kwargs = make_kwargs(arch, cfg)

    print("=" * 70)
    print(f"SO SANH: {arch} | {COMPARE_EPOCHS} epoch")
    print("=" * 70)
    set_seed(cfg["seed"])
    m = build_model(arch, **kwargs)
    res = train_loop(m, LOADERS, cfg, run_name=f"cmp_{arch}", verbose=True)
    compare_models[arch] = res["model"]

    metrics, _, _ = evaluate_model(res["model"], LOADERS["test"], SCALER)
    compare_results.append({
        "arch": arch,
        "mae": metrics["mae"],
        "rmse": metrics["rmse"],
        "mae_h1": metrics["mae_h1"],
        "mae_h24": metrics["mae_h24"],
        "mae_h72": metrics["mae_h72"],
        "mae_h144": metrics["mae_h144"],
        "params": count_parameters(res["model"]),
    })

COMPARE_DF = pd.DataFrame(compare_results)
print("\n" + "=" * 70)
print("BANG SO SANH 3 KIEN TRUC (tap kiem thu 2016, tam 144)")
print("=" * 70)
display(COMPARE_DF.round(4))
COMPARE_DF.to_csv("results/compare_architectures.csv", index=False, encoding="utf-8")
print("Đã lưu results/compare_architectures.csv")

# Biểu đồ so sánh
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
x = np.arange(len(COMPARE_DF))
ax.bar(x - 0.2, COMPARE_DF["mae"], 0.4, label="MAE", color="steelblue")
ax.bar(x + 0.2, COMPARE_DF["rmse"], 0.4, label="RMSE", color="indianred")
ax.axhline(THRESHOLDS["mae"], ls="--", color="navy", label="nguong MAE 1.5")
ax.set_xticks(x)
ax.set_xticklabels(COMPARE_DF["arch"], rotation=20, ha="right")
ax.set_ylabel("°C")
ax.set_title("MAE / RMSE theo kien truc")
ax.legend()
ax.grid(alpha=0.3, axis="y")

ax = axes[1]
for _, r in COMPARE_DF.iterrows():
    ax.plot(HORIZONS, [r[f"mae_h{h}"] for h in HORIZONS], "o-", label=r["arch"], ms=5)
ax.set_xlabel("buoc du bao (10 phut)")
ax.set_ylabel("MAE (°C)")
ax.set_title("Sai so theo tam du bao")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# === 19. Đo độ trễ suy luận (ngưỡng ≤ 100 ms/mẫu) ===

def measure_latency(model, sample_x, repeats=30):
    """Đo trung bình thời gian suy luận 1 mẫu/ngày (ms)."""
    model.eval()
    x = sample_x.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        for _ in range(5):
            model(x, y_in=None)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(repeats):
            model(x, y_in=None)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - t0
    return elapsed / repeats * 1000


sample_x = next(iter(LOADERS["test"]))["input"][0]
latency_rows = []
for arch, mdl in compare_models.items():
    ms = measure_latency(mdl, sample_x)
    latency_rows.append({"arch": arch, "latency_ms": round(ms, 2),
                         "dat": ms <= 100})
    print(f"{arch:<18} {ms:7.2f} ms/mẫu  -> {'ĐẠT' if ms <= 100 else 'KHÔNG ĐẠT'} (≤ 100)")

latency_df = pd.DataFrame(latency_rows)
latency_df.to_csv("results/latency.csv", index=False, encoding="utf-8")
print("\nĐã lưu results/latency.csv")


In [ ]:
# === 20. Tổng kết và kiểm tra nghiệm thu ===

print("=" * 74)
print("TONG KET KET QUA")
print("=" * 74)

print("\n[1] Cau hinh tot nhat (sweep tren tap kiem dinh 2015):")
print(f"    {BEST_NAME} | val MAE = {BEST_ROW['val_mae']:.4f} °C | "
      f"best epoch {BEST_ROW['best_epoch']}")

print("\n[2] Ket qua kiem thu 2016 cua luan chay chinh:")
for k, label, thr in (("mae", "MAE toan tang", THRESHOLDS["mae"]),
                      ("rmse", "RMSE toan tang", THRESHOLDS["rmse"]),
                      ("mae_h24", "MAE@24", THRESHOLDS["mae_h24"]),
                      ("mae_h144", "MAE@144", THRESHOLDS["mae_h144"])):
    v = final_metrics[k]
    print(f"    {label:<16}{v:8.4f} °C  (≤ {thr})  -> {'ĐẠT' if v <= thr else 'KHÔNG ĐẠT'}")
for k, label in (("persistence_mae", "Persistence"),
                 ("seasonal_naive_mae", "Seasonal-naive")):
    beat = final_metrics["mae"] < final_metrics[k]
    print(f"    {label:<16}{final_metrics[k]:8.4f} °C  -> "
          f"{'ĐẠT (đã vượt)' if beat else 'KHÔNG ĐẠT'}")
print(f"    MAE theo tầm: " + " | ".join(
    f"@{h} = {final_metrics[f'mae_h{h}']:.3f} °C" for h in HORIZONS))
print(f"    Giá trị ngoài biên vật lý: {final_metrics['out_of_bounds_count']}")

print("\n[3] So sanh 3 kien truc (tap kiem thu 2016):")
display(COMPARE_DF.round(4))

print("\n[4] Do tre suy luan:")
display(latency_df)

print("\n[5] File ket qua da luu:")
for p in ("results/sweep_results.csv", "results/compare_architectures.csv",
          "results/latency.csv", "results/final_metrics.json", "results/final_model.pt"):
    print(f"    {p}")

print("\n[6] So sanh voi moc EDA:")
print(f"    persistence EDA 2016   : {PERSISTENCE_2016:.4f} °C")
print(f"    seasonal-naive EDA 2016: {SEASONAL_2016:.4f} °C")
print(f"    mo hinh cua ban        : {final_metrics['mae']:.4f} °C")


In [ ]:
# === 21. Lưu kết quả lên Google Drive (không mất khi phiên Colab kết thúc) ===

# Mount Drive để lưu kết quả lâu dài. Nếu không muốn dùng Drive, bỏ qua ô này.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/jena-seq2seq")
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    for name in ("sweep_results.csv", "compare_architectures.csv",
                 "latency.csv", "final_metrics.json", "final_model.pt"):
        src = Path("results") / name
        if src.exists():
            import shutil
            shutil.copy2(src, DRIVE_DIR / name)
    print("Đã lưu kết quả lên Drive:", DRIVE_DIR)
    for p in sorted(DRIVE_DIR.iterdir()):
        print(f"  {p.name}  ({p.stat().st_size // 1024} KB)")
except Exception as exc:  # noqa: BLE001
    print("Không mount được Drive (bỏ qua ô này):", exc)

print("\nKết quả vẫn nằm trong thư mục results/ của phiên Colab hiện tại.")
